# Week 4: Composition — Building with Objects
### PHASE 2: Composing Components

*📚 Object Oriented Programming · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

1. Explain what **composition** means (the "has-a" relationship)
2. Create objects that **contain other objects**
3. Build a **sensor system** from separate components
4. Understand why composition is better than putting everything in one big class
5. **Wire components together** so they communicate
6. Recognize common **composition patterns** in engineering

## 🎯 Core Mastery Connection

**THIS IS THE CORE WEEK.** Composition means one object contains others. A `Sensor` + `Filter` + `Logger` wired into a `MonitoringSystem` — that is the heart of OOP design. Everything you learned in Weeks 1-3 (creating classes, adding behavior, protecting state) was preparation for this moment. Now you combine small, reliable components into a working system.

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import matplotlib.pyplot as plt

---
## Part 1: What is Composition?

**Composition** means building complex objects by combining simpler objects.

Think of a **robot arm**:
- It **has a** motor
- It **has a** sensor
- It **has a** gripper

Each part is its own object. The robot arm **contains** them.

This is the **"has-a"** relationship:

| Relationship | Meaning | Example |
|---|---|---|
| **has-a** (Composition) | One object contains another | A Car **has a** Engine |
| **is-a** (Inheritance) | One object is a type of another | A Car **is a** Vehicle |

This week we focus on **"has-a"**. We will learn "is-a" (inheritance) later.

**Why is this useful?**
- Each part is **simple** on its own
- Parts can be **reused** in different systems
- You can **test** each part separately
- You can **swap** one part for another

---
## Part 2: Objects Inside Objects

The simplest form of composition: store one object inside another.

**Figure 4.1** — A motor and a robot that contains it:

In [ ]:
# Figure 4.1 - Basic composition: Robot has a Motor

class Motor:
    def __init__(self, name):
        self.name = name
        self._speed = 0

    @property
    def speed(self):
        return self._speed

    @speed.setter
    def speed(self, value):
        if 0 <= value <= 100:
            self._speed = value

    def status(self):
        return f"Motor '{self.name}': {self._speed}%"


class Robot:
    def __init__(self, name):
        self.name = name
        # Composition: Robot HAS-A Motor (created inside)
        self.left_motor = Motor("Left")
        self.right_motor = Motor("Right")

    def move_forward(self, speed):
        """Both motors go at the same speed."""
        self.left_motor.speed = speed
        self.right_motor.speed = speed
        print(f"{self.name} moving forward at {speed}%")

    def status(self):
        print(f"--- {self.name} ---")
        print(f"  {self.left_motor.status()}")
        print(f"  {self.right_motor.status()}")


# Test
robot = Robot("Explorer")
robot.move_forward(60)
robot.status()

The `Robot` does not know **how** the motor works internally. It just calls `motor.speed = value`. This is the beauty of composition!

**Figure 4.2** — You can also pass objects in from outside:

In [ ]:
# Figure 4.2 - Passing objects in (dependency injection)

class Motor:
    def __init__(self, name, max_speed=100):
        self.name = name
        self._speed = 0
        self._max_speed = max_speed

    @property
    def speed(self):
        return self._speed

    @speed.setter
    def speed(self, value):
        self._speed = max(0, min(value, self._max_speed))


class Car:
    def __init__(self, name, motor):
        self.name = name
        self.motor = motor   # Motor is passed in, not created inside

    def drive(self, speed):
        self.motor.speed = speed
        print(f"{self.name} driving at {self.motor.speed}%")


# Create motor separately, then give it to the car
fast_motor = Motor("V8", max_speed=100)
slow_motor = Motor("Eco", max_speed=50)

sports_car = Car("Sports Car", fast_motor)
eco_car = Car("Eco Car", slow_motor)

sports_car.drive(80)
eco_car.drive(80)   # Will be capped at 50

| Approach | Code | When to Use |
|---|---|---|
| Create inside | `self.motor = Motor("M1")` | Object always needs the same type of part |
| Pass in | `self.motor = motor` | You want to choose which part to use |

---
## Part 3: Building a Sensor System

Let's build a real example step by step. We want a **temperature monitoring system** with three parts:

1. **Sensor** — reads temperature values
2. **Filter** — smooths out noisy readings
3. **Logger** — records the data

We will build each part as a separate class, then combine them.

**Figure 4.3** — The Sensor component:

In [ ]:
# Figure 4.3 - Sensor component

class TemperatureSensor:
    """Reads temperature values."""

    def __init__(self, name):
        self.name = name
        self._reading = 0.0

    @property
    def reading(self):
        return self._reading

    def measure(self, value):
        """Simulate a new measurement."""
        self._reading = value
        return self._reading

# Test alone
sensor = TemperatureSensor("Engine")
sensor.measure(45.2)
print(f"Sensor reading: {sensor.reading}C")

---
## Part 4: The Filter Component

Real sensors give noisy readings. A **filter** smooths them out by averaging the last N values.

**Figure 4.4** — The Filter component:

In [ ]:
# Figure 4.4 - Filter component (moving average)

class MovingAverageFilter:
    """Smooths values by averaging the last N readings."""

    def __init__(self, window_size=3):
        self._window_size = window_size
        self._values = []   # stores recent values

    @property
    def window_size(self):
        return self._window_size

    def add(self, value):
        """Add a new value and return the filtered result."""
        self._values.append(value)
        # Keep only the last N values
        if len(self._values) > self._window_size:
            self._values.pop(0)
        # Return the average
        return sum(self._values) / len(self._values)

    def reset(self):
        """Clear all stored values."""
        self._values = []

# Test alone
f = MovingAverageFilter(window_size=3)
print(f.add(10))   # Average of [10] = 10.0
print(f.add(20))   # Average of [10, 20] = 15.0
print(f.add(30))   # Average of [10, 20, 30] = 20.0
print(f.add(40))   # Average of [20, 30, 40] = 30.0

---
## Part 5: The Logger Component

A **logger** records values so we can look at the history later.

**Figure 4.5** — The Logger component:

In [ ]:
# Figure 4.5 - Logger component

class DataLogger:
    """Records values with labels."""

    def __init__(self):
        self._log = []   # list of (label, value) pairs

    @property
    def count(self):
        """How many entries have been logged."""
        return len(self._log)

    def record(self, label, value):
        """Add an entry to the log."""
        self._log.append((label, round(value, 2)))

    def show(self):
        """Print all logged entries."""
        print(f"--- Log ({self.count} entries) ---")
        for i, (label, value) in enumerate(self._log):
            print(f"  [{i}] {label}: {value}")

    def clear(self):
        """Clear the log."""
        self._log = []

# Test alone
log = DataLogger()
log.record("temp", 25.3)
log.record("temp", 26.1)
log.show()

---
## Part 6: Wiring Components Together

Now we combine all three parts into a **MonitoringSystem**.

The system:
1. Reads from the **sensor**
2. Smooths the value through the **filter**
3. Saves the result in the **logger**

**Figure 4.6** — The complete monitoring system:

In [ ]:
# Figure 4.6 - Composing all parts into a system

class MonitoringSystem:
    """Combines sensor + filter + logger into one system."""

    def __init__(self, sensor_name, filter_window=3):
        # Composition: this system HAS-A sensor, filter, and logger
        self.sensor = TemperatureSensor(sensor_name)
        self.filter = MovingAverageFilter(filter_window)
        self.logger = DataLogger()

    def process(self, raw_value):
        """Take a raw reading, filter it, and log it."""
        # Step 1: Sensor gets the raw value
        self.sensor.measure(raw_value)

        # Step 2: Filter smooths it
        filtered = self.filter.add(raw_value)

        # Step 3: Logger records it
        self.logger.record(self.sensor.name, filtered)

        return filtered

    def report(self):
        """Show the full log."""
        print(f"=== {self.sensor.name} Monitoring ===")
        self.logger.show()


# Use the system
system = MonitoringSystem("Engine Temp", filter_window=3)

# Simulate noisy temperature readings
readings = [44.5, 46.2, 45.0, 47.8, 46.5, 48.1, 45.9]

for raw in readings:
    filtered = system.process(raw)
    print(f"Raw: {raw:5.1f}  ->  Filtered: {filtered:.2f}")

print()
system.report()

**Notice how clean this is!** Each class does one job:

| Component | Job |
|---|---|
| `TemperatureSensor` | Store a raw reading |
| `MovingAverageFilter` | Smooth noisy values |
| `DataLogger` | Record history |
| `MonitoringSystem` | Wire the parts together |

**Figure 4.7** — You can swap components easily:

In [ ]:
# Figure 4.7 - Swapping components

# System with a small filter window (responds quickly to changes)
fast_system = MonitoringSystem("Exhaust", filter_window=2)

# System with a large filter window (very smooth, but slow)
smooth_system = MonitoringSystem("Coolant", filter_window=5)

readings = [30, 35, 32, 38, 33, 36, 31]

print("Fast filter (window=2) vs Smooth filter (window=5):")
print(f"{'Raw':>5}  {'Fast':>8}  {'Smooth':>8}")
print("-" * 25)

for raw in readings:
    fast = fast_system.process(raw)
    smooth = smooth_system.process(raw)
    print(f"{raw:5.1f}  {fast:8.2f}  {smooth:8.2f}")

---
## Part 6b: Visualizing Raw vs Filtered Data

In CP2 you learned how to use `matplotlib` to plot data. Let's use it here to **see** the effect of composition — how the filter smooths the raw sensor readings.

**Figure 4.7b** — Plotting raw vs filtered data:

In [ ]:
# Figure 4.7b - Visualizing composition: raw vs filtered

import matplotlib.pyplot as plt

# Create two systems with different filter windows
fast_sys = MonitoringSystem("Engine", filter_window=2)
smooth_sys = MonitoringSystem("Engine", filter_window=5)

readings = [44.5, 46.2, 45.0, 47.8, 46.5, 48.1, 45.9, 44.2, 47.0, 46.8,
            45.5, 48.3, 44.8, 46.0, 47.5, 45.3, 48.0, 46.2, 44.9, 47.1]

fast_filtered = [fast_sys.process(r) for r in readings]
smooth_filtered = [smooth_sys.process(r) for r in readings]

plt.figure(figsize=(10, 5))
plt.plot(readings, 'o--', color='gray', alpha=0.5, label='Raw readings')
plt.plot(fast_filtered, 's-', color='steelblue', label='Filter window=2')
plt.plot(smooth_filtered, 'd-', color='tomato', label='Filter window=5')
plt.xlabel('Reading #')
plt.ylabel('Temperature (°C)')
plt.title('Composition in Action: Raw vs Filtered Sensor Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The plot clearly shows how the filter **smooths out the noise**. A larger window (5) gives a smoother curve but reacts more slowly to real changes. This is a classic engineering trade-off!

Notice that the plot was easy to create because our components are well-separated: the `MonitoringSystem` handles the logic, and we just collect the results for plotting.

---
## Part 7: Benefits of Composition

Why not put everything in one big class?

| One Big Class | Composition |
|---|---|
| Hard to understand | Each part is simple |
| Hard to test | Test each part alone |
| Can't reuse parts | Reuse parts in other systems |
| Change one thing, break everything | Change one part, others stay the same |
| 200 lines in one class | 50 lines in 4 small classes |

**Figure 4.8** — Reusing the same components in a different system:

In [ ]:
# Figure 4.8 - Reusing components in a different system

class AlertSystem:
    """Monitors a sensor and raises an alert if value is too high."""

    def __init__(self, sensor_name, threshold):
        # Reuses the SAME sensor and logger classes
        self.sensor = TemperatureSensor(sensor_name)
        self.logger = DataLogger()
        self._threshold = threshold
        self._alert_active = False

    @property
    def alert_active(self):
        return self._alert_active

    def check(self, value):
        """Check a reading against the threshold."""
        self.sensor.measure(value)
        self.logger.record(self.sensor.name, value)

        if value > self._threshold:
            self._alert_active = True
            print(f"ALERT: {self.sensor.name} is {value}C "
                  f"(threshold: {self._threshold}C)")
        else:
            self._alert_active = False

# Test
alert = AlertSystem("CPU Temp", threshold=75)

alert.check(60)   # OK
alert.check(72)   # OK
alert.check(80)   # ALERT!
alert.check(68)   # OK again

We reused `TemperatureSensor` and `DataLogger` without changing them at all!

---
## Part 8: Common Patterns

Here are common ways you will see composition used:

**Pattern 1: Container** — One object holds a list of other objects

**Figure 4.9** — A robot with multiple sensors:

In [ ]:
# Figure 4.9 - Container pattern: list of objects

class SensorArray:
    """Holds multiple sensors and reads them all at once."""

    def __init__(self):
        self._sensors = []    # list of sensor objects

    def add_sensor(self, sensor):
        """Add a sensor to the array."""
        self._sensors.append(sensor)
        print(f"Added sensor: {sensor.name}")

    @property
    def count(self):
        return len(self._sensors)

    def read_all(self):
        """Print readings from all sensors."""
        print(f"--- Sensor Array ({self.count} sensors) ---")
        for s in self._sensors:
            print(f"  {s.name}: {s.reading}C")

# Build a sensor array
array = SensorArray()

s1 = TemperatureSensor("Engine")
s2 = TemperatureSensor("Exhaust")
s3 = TemperatureSensor("Cabin")

array.add_sensor(s1)
array.add_sensor(s2)
array.add_sensor(s3)

# Simulate readings
s1.measure(85.0)
s2.measure(120.5)
s3.measure(22.3)

array.read_all()

**Pattern 2: Coordinator** — One object manages how others interact

**Figure 4.10** — A simple LED controller:

In [ ]:
# Figure 4.10 - Coordinator pattern

class LED:
    def __init__(self, color):
        self.color = color
        self._is_on = False

    @property
    def is_on(self):
        return self._is_on

    def turn_on(self):
        self._is_on = True

    def turn_off(self):
        self._is_on = False

    def status(self):
        state = "ON" if self._is_on else "OFF"
        return f"{self.color} LED: {state}"


class TrafficLight:
    """Coordinates three LEDs."""

    def __init__(self):
        self.red = LED("Red")
        self.yellow = LED("Yellow")
        self.green = LED("Green")
        self._state = "stop"

    def set_state(self, state):
        """Change the traffic light state."""
        # Turn all off first
        self.red.turn_off()
        self.yellow.turn_off()
        self.green.turn_off()

        # Turn on the right one
        if state == "stop":
            self.red.turn_on()
        elif state == "caution":
            self.yellow.turn_on()
        elif state == "go":
            self.green.turn_on()

        self._state = state

    def status(self):
        print(f"Traffic Light ({self._state}):")
        print(f"  {self.red.status()}")
        print(f"  {self.yellow.status()}")
        print(f"  {self.green.status()}")


# Test
light = TrafficLight()
light.set_state("stop")
light.status()
print()
light.set_state("go")
light.status()

| Pattern | Description | Example |
|---|---|---|
| **Container** | Holds a list of similar objects | SensorArray holds many Sensors |
| **Coordinator** | Manages how parts work together | TrafficLight controls 3 LEDs |
| **Pipeline** | Data flows from one part to the next | Sensor -> Filter -> Logger |

---
## 🎢 Exercises — Composing Systems from Components

> **Core mastery in practice:** Every exercise below asks you to build a system from multiple objects. Think about the "has-a" relationship: which object *owns* which? Each component should do one job well. The composed system wires them together and coordinates their work. This is the skill at the heart of this course.

Complete the exercises below. Each exercise has a difficulty level:
- **Easy** — Direct application of what you learned
- **Medium** — Requires some thinking
- **Challenge** — Combines multiple concepts

### Exercise 1 (Easy)
Create a `Battery` class (with `charge` and `use(amount)` method) and a `Flashlight` class that **has a** `Battery`. The `Flashlight` should have a `turn_on()` method that only works if the battery charge is above 0.

<details>
<summary>💡 Hint</summary>
The Flashlight <code>__init__</code> takes a Battery object: <code>self._battery = battery</code>. <code>turn_on()</code> checks <code>self._battery.charge > 0</code>.
</details>

In [ ]:
# ✏️ [EX1] Flashlight has-a Battery



### Exercise 2 (Easy)
Create a `Wheel` class (with `diameter` and `rotate(degrees)` method) and a `Bicycle` class that **has two** `Wheel` objects (front and rear). Add a `pedal()` method that rotates both wheels.

<details>
<summary>💡 Hint</summary>
The Bicycle stores two Wheel objects. Use <code>self._front = Wheel(...)</code> and <code>self._rear = Wheel(...)</code>.
</details>

In [ ]:
# ✏️ [EX2] Bicycle has two Wheels



### Exercise 3 (Easy)
Create a `Speaker` class (with `play(sound)` method) and a `Buzzer` class that **has a** `Speaker`. The `Buzzer` should have `beep()` and `alarm()` methods that tell the speaker to play different sounds.

<details>
<summary>💡 Hint</summary>
The Buzzer owns a Speaker. <code>beep()</code> calls <code>self._speaker.play(frequency)</code>.
</details>

In [ ]:
# ✏️ [EX3] Buzzer has-a Speaker



### Exercise 4 (Easy)
Create a `GPS` class with `latitude` and `longitude` properties, and a `Drone` class that **has a** `GPS`. The `Drone` should have a `location()` method that prints the GPS coordinates.

<details>
<summary>💡 Hint</summary>
The Drone has a GPS object. <code>fly_to(x, y)</code> updates the GPS position. <code>get_position()</code> reads from the GPS.
</details>

In [ ]:
# ✏️ [EX4] Drone has-a GPS



### Exercise 5 (Medium)
Create a `Motor` class and a `Sensor` class. Then create a `ConveyorBelt` class that **has a** `Motor` and **has a** `Sensor`. The sensor detects objects (has a `detect()` method that returns True/False). When an object is detected, the conveyor belt should start the motor. When nothing is detected, it should stop.

<details>
<summary>💡 Hint</summary>
The ConveyorBelt has a Motor and a list of sensors. <code>start()</code> starts the motor, <code>check()</code> reads all sensors.
</details>

In [ ]:
# ✏️ [EX5] ConveyorBelt with Motor and Sensor



### Exercise 6 (Medium)
Create a `MotorController` class that **has a** `Motor` and **has a** `TemperatureSensor`. Add a `run(speed)` method that checks the temperature first. If temperature > 80C, it should reduce speed to 50% to prevent overheating.

<details>
<summary>💡 Hint</summary>
Store <code>self._motors = [motor1, motor2]</code>. <code>set_speed()</code> applies to all motors. <code>stop()</code> stops all.
</details>

In [ ]:
# ✏️ [EX6] MotorController with safety check



### Exercise 7 (Medium)
Create a `Playlist` class that stores a list of song names (strings). Create a `MusicPlayer` class that **has a** `Playlist` and **has a** `Speaker`. The player should have `play_next()` and `status()` methods.

<details>
<summary>💡 Hint</summary>
MusicPlayer has a Playlist and a Speaker. <code>play()</code> gets the current song from playlist and sends to speaker.
</details>

In [ ]:
# ✏️ [EX7] MusicPlayer with Playlist and Speaker



### Exercise 8 (Medium)
Create a `SensorHub` class that can hold **any number** of `TemperatureSensor` objects (use a list). Add methods:
- `add_sensor(name)` — creates and adds a new sensor
- `read_all()` — returns a dictionary of {name: reading}
- `average()` — returns the average of all sensor readings

<details>
<summary>💡 Hint</summary>
SensorHub stores a list of sensors. <code>add_sensor()</code> appends. <code>read_all()</code> returns a dict of sensor_name: value pairs.
</details>

In [ ]:
# ✏️ [EX8] SensorHub with dynamic sensor list



### Exercise 9 (Challenge)
Build a `SmartHome` system with:
- A `Thermostat` class (has target_temp and current_temp)
- A `Heater` class (can be on/off, has a power level 0-100)
- A `SmartHome` class that **has a** `Thermostat` and **has a** `Heater`

The `SmartHome` should have a `regulate()` method that:
- Turns the heater ON if current_temp < target_temp
- Sets heater power based on the difference (bigger difference = more power)
- Turns the heater OFF if current_temp >= target_temp

<details>
<summary>💡 Hint</summary>
SmartHome has Room objects, each Room has Device objects. Use nested composition: <code>home.rooms[0].devices</code>.
</details>

In [ ]:
# ✏️ [EX9] SmartHome with Thermostat and Heater



### Exercise 10 (Challenge)
Build a `DataPipeline` class that:
- Has a `TemperatureSensor`
- Has a `MovingAverageFilter`
- Has a `DataLogger`
- Has an `AlertSystem` (new class: checks if filtered value exceeds a threshold and prints a warning)

The pipeline should process data through all four stages: sense -> filter -> check alert -> log

<details>
<summary>💡 Hint</summary>
Each pipeline stage is an object with a <code>process(data)</code> method. The pipeline calls them in sequence.
</details>

In [ ]:
# ✏️ [EX10] DataPipeline with four components



### Exercise 11 (Challenge)
Build a `RobotArm` class composed of:
- 3 `ServoMotor` objects (base, elbow, wrist) — each with angle 0-180
- A `Gripper` object (opening 0-100mm)

Add methods:
- `home()` — set all servos to 90 degrees, gripper to 100mm (open)
- `pick(base_angle, elbow_angle, wrist_angle)` — move to position and close gripper
- `place(base_angle, elbow_angle, wrist_angle)` — move to position and open gripper
- `status()` — print all component states

<details>
<summary>💡 Hint</summary>
RobotArm has Joint objects. Each Joint has position/angle. <code>move_to()</code> sets all joints. Use a list of joints.
</details>

In [ ]:
# ✏️ [EX11] RobotArm with servos and gripper



### Exercise 12 (Challenge)
Build a `WeatherStation` class composed of:
- `TemperatureSensor` (readings in Celsius)
- `PressureSensor` class (readings in hPa, range 900-1100)
- `DataLogger`

Add a `measure(temp, pressure)` method that logs both values, and a `forecast()` method that prints:
- "Sunny" if pressure > 1013 and temp > 20
- "Rainy" if pressure < 1000
- "Cloudy" otherwise

<details>
<summary>💡 Hint</summary>
WeatherStation composes Sensor objects for temp, humidity, pressure. <code>report()</code> reads all and formats output.
</details>

In [ ]:
# ✏️ [EX12] WeatherStation with forecast



---
### 🌉 Bridge to Next Week

This week we learned how to **build complex systems from simple parts** using composition.

Key ideas:
- Objects can **contain** other objects ("has-a" relationship)
- Each part does **one job** well
- Parts can be **reused** and **swapped**

Next week, we will learn about **Inheritance** — the "is-a" relationship. Instead of one object containing another, we will see how one class can be **based on** another class and extend its behavior.

See you in **Week 5**!

---
## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_04"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")